In [ ]:
import pandas as pd
import re

#Raw RPK matrix: peptides x sample IDs

cohort_df = pd.read_csv("./preprocessing/earliest_tp_rpk_avg_value_for_cohort_techreps_071626.csv").set_index("peptide")

#sample metadata with time info
meta = pd.read_csv("./preprocessing/master_sample_metadata_060626.csv")

#import df_hbdb
hbdb_df = pd.read_csv("./preprocessing/hbdb_pass_100k_techreps_averaged_071626.csv").set_index("peptide")

#load peptide protein metadata
protein_metadata = pd.read_csv("lassa_library_sequences_species_standardized_070726_ICTV_corrected.csv")
protein_metadata = protein_metadata.set_index('seq_id')

In [5]:
df_concat = pd.concat([cohort_df, hbdb_df], axis=1)

exposed_cols = cohort_df.columns
hbdb_cols = hbdb_df.columns

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

#prefilter = peptides nobody binds to can't be significant. keep peptides seen in at least 5%

keep = (df_concat > 0 ).mean(axis=1) >= 0.05
sub = df_concat.loc[keep]

A = sub[exposed_cols].to_numpy()
B = sub[hbdb_cols].to_numpy()

#axis = 1 means run the test along each row, so one test per peptide, all ~90k at once!
#res.pvalue and res.statistic come back as arrays, one entry per peptide
#asymtpotit assumes that the null distribution of the U statistic is approx. normal, 
#not that the data is normal! Central limit theorem applies that the null distribution of U converges to a normal
res = mannwhitneyu(A, B, axis=1, alternative='two-sided', method='asymptotic')

#multiple testing
q = multipletests(res.pvalue, method='fdr_bh')[1]

#effect size. rank a biserial correlation, rescales U onto -1 o +1. 
#+1 means every exposed sample is above every control, 0 means no separation, 
#-1 the reverse. fraction of exposed wins 
rbc = 2 * res.statistic / (A.shape[1] * B.shape[1]) -1 

In [ ]:
import numpy as np

res_df = pd.DataFrame({
    "p": res.pvalue, 
    "q": q, 
    "rbc": rbc, 
    "median_exposed": np.median(A, axis=1), 
    'median_healthy': np.median(B, axis=1), 
}, 
index=sub.index).sort_values("p")

print(f"{(res_df.q < 0.05).sum()} peptides at 1 > 0.05")

57765 peptides at 1 > 0.05


In [43]:
#filtering on effect size
strong = res_df[(res_df.q < 0.01) & (res_df.rbc.abs() > 0.5)]
print(f"{len(strong)} peptides with q<0.01 and |rbc|> 0.5")

113 peptides with q<0.01 and |rbc|> 0.5


In [80]:
for r in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    n = ((res_df.q < 0.05) & (res_df.rbc > r)).sum()
    print(f"|rbc| > {r}: {n} peptides")

|rbc| > 0.2: 826 peptides
|rbc| > 0.3: 287 peptides
|rbc| > 0.4: 104 peptides
|rbc| > 0.5: 52 peptides
|rbc| > 0.6: 19 peptides
|rbc| > 0.7: 9 peptides
|rbc| > 0.8: 6 peptides


In [81]:
strong = res_df[(res_df.q < 0.05) & (res_df.rbc > 0.3)]

In [82]:
strong.head(3)

,p,q,rbc,median_exposed,median_healthy
peptide,,,,,
PF3D7_0501400.1-p1|transcript=PF3D7_0501400.1|organism=Plasmodium_falciparum_3D7|gene_product=interspersed_repeat_antigen|tile_1,2.901345e-47,2.363030e-42,0.987840,1823.279911,0.274787
PF3D7_0801000.1-p1|transcript=PF3D7_0801000.1|organism=Plasmodium_falciparum_3D7|gene_product=Plasmodium_exported_protein_(PHISTc)-_unknown_function|tile_1,7.532849e-45,3.067602e-40,0.961377,928.784659,1.145048
PF3D7_0220000.1-p1|transcript=PF3D7_0220000.1|organism=Plasmodium_falciparum_3D7|gene_product=liver_stage_antigen_3|tile_1,5.359853e-42,1.455129e-37,0.929008,794.535030,0.612346


## upper quartile normalization - so every sample gets normalized by division by its own 75th percentile. 
### prevents that the signal is eaten up by only the most dominant peptides - in case that is the goal. rerun statistics on this dataframe

In [85]:
strong  = strong.join(protein_metadata[['sequence', 'accession', 'protein_name',
       'order', 'family', 'genus', 'species', 'isolate', 'geo_loc_name',
       'protein_std', 'fragment']])
strong [["species", "protein_name", "fragment", "rbc", "q", "median_exposed", "median_healthy"]]

ValueError: columns overlap but no suffix specified: Index(['sequence', 'accession', 'protein_name', 'order', 'family', 'genus',
       'species', 'isolate', 'geo_loc_name', 'protein_std', 'fragment'],
      dtype='object')

In [86]:
strong = strong.sort_values(['species', 'protein_name', 'fragment'], ascending=False)

strong.loc[strong['species'] == 'Orthonairovirus haemorrhagiae']

,p,q,rbc,median_exposed,median_healthy,sequence,accession,protein_name,order,family,genus,species,isolate,geo_loc_name,protein_std,fragment
peptide,,,,,,,,,,,,,,,,
QKW94271.1|precursor|Orthonairovirus_haemorrhagiae|isolate=MCL-19-H-2077|Homo_sapiens|India:_Gujarat_State|2019|segment=M|strain=?|tile_10,1.569900e-07,2.050841e-06,0.358812,10.726904,6.270070,LKQLATSGSTIPPTQTVLPQSGTSIAIQDIYLSPTNRSKRNLDMEI,QKW94271.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,MCL-19-H-2077,India: Gujarat State,glycoprotein,10
QQL93572.1|precursor|Orthonairovirus_haemorrhagiae|isolate=?|Hyalomma_lusitanicum|Spain|?|segment=M|strain=Badajoz_2018|tile_8,1.653500e-12,3.497948e-10,0.482672,2.188285,0.612276,TPSRSGKESSATNSPHPASHRPSTPPATAQEPIGNNSHNATEHLGS,QQL93572.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,NaN,Spain,glycoprotein,8
QKW94271.1|precursor|Orthonairovirus_haemorrhagiae|isolate=MCL-19-H-2077|Homo_sapiens|India:_Gujarat_State|2019|segment=M|strain=?|tile_8,4.686013e-08,8.204150e-07,0.373230,1.469016,0.544761,TPSKSSKGSSATNSPHPAASRPTTPSTTAQKSTDNNSHNATEQLES,QKW94271.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,MCL-19-H-2077,India: Gujarat State,glycoprotein,8
WLV27569.1|precursor|Orthonairovirus_haemorrhagiae|isolate=?|?|China|1988|segment=M|strain=88168|tile_5,6.061501e-14,2.697732e-11,0.512956,2.316397,0.563749,TLPETTLEPLTTTDTDASSTVVVDVSTQTVKDTPISTTYTSLPSTP,WLV27569.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,NaN,China,glycoprotein,5
XKB88840.1|precursor|Orthonairovirus_haemorrhagiae|isolate=CCHF/NIHPAK-37/2024|Homo_sapiens|Pakistan:_Punjab|12-Jul-2024|segment=M|strain=?|tile_4,1.599216e-17,2.553917e-14,0.582906,12.340423,2.916106,TPDTQITTDSGSGESSTDPSVTNKDTSPTSVAIVQTQETEAPTTLN,XKB88840.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,CCHF/NIHPAK-37/2024,Pakistan: Punjab,glycoprotein,4
WLV27569.1|precursor|Orthonairovirus_haemorrhagiae|isolate=?|?|China|1988|segment=M|strain=88168|tile_4,7.979545e-13,2.005871e-10,0.472654,0.601906,0.000000,VSVSGSEGSGDTNTSPSVTTQGPTLPETTLEPLTTTDTDASSTVVV,WLV27569.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,NaN,China,glycoprotein,4
XQQ58331.1|precursor|Orthonairovirus_haemorrhagiae|isolate=CCHFV_N.O.34|Homo_sapiens|Tajikistan|07-Jul-2017|segment=M|strain=?|tile_4,2.386994e-08,4.897006e-07,0.381048,1.314961,0.460753,ASVNGPEGSGDTNTPPPITTQGLTPSETTLELLTTTDIDASSTVVV,XQQ58331.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,CCHFV_C17,Tajikistan,glycoprotein,4
XMH26459.1|precursor|Orthonairovirus_haemorrhagiae|isolate=?|Hyalomma_impeltatum|Senegal|Sep-2022|segment=M|strain=ARD407551|tile_4,5.298127e-08,8.997315e-07,0.366919,0.604191,0.123021,NTPDTQVTTDSGSGEPSTDPQTTSREVPSTSDATIKTQESEMLRAQ,XMH26459.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,NaN,Senegal,glycoprotein,4
ADV31331.1|precursor|Orthonairovirus_haemorrhagiae|isolate=?|Homo_sapiens|Bulgaria|1981|segment=M|strain=V42/81|tile_4,2.812674e-07,3.199456e-06,0.351139,1.893218,0.799146,TPTSETEGSGEATPLSNTIQAPSSPETTSERPATAVISTSGTDSTN,ADV31331.1,glycoprotein precursor,Hareavirales,Nairoviridae,Orthonairovirus,Orthonairovirus haemorrhagiae,NaN,Bulgaria,glycoprotein,4


In [ ]:
uq = df_concat[df_concat > 0].quantile(0.75)  #masks zeros to Nan, then upper quartile scaling per sampling. 
norm = df_concat.div(uq, axis=1) * uq.median()

In [89]:
norm

,C-109-3_4_E7,C-111-3_3_I6,C-112-1_3_B3,C-130-2_3_C3,C-135-2_1_D2,C-140-1_1_A7,C-144-2_2_B5,C-147-2_1_F5,C-150-2_1_H3,C-183-1_1_F4,...,HBDB-097,HBDB-098,HBDB-099,HBDB-100,HBDB-102,HBDB-106,HBDB-107,HBDB-109,HBDB-111,HBDB-115
peptide,,,,,,,,,,,,,,,,,,,,,
WFG38034.1|precursor|Aba-Mianyang_virus|SC/C3-30.18/2021|Ochotona_sp.|China|Aug-2021|S|?|tile_1,0.000000,0.483343,0.478534,0.000000,0.000000,0.422960,0.581869,0.580288,0.000000,0.000000,...,0.169296,0.000000,0.162931,0.000000,0.000000,0.452256,0.000000,0.000000,0.155336,0.000000
WFG38034.1|precursor|Aba-Mianyang_virus|SC/C3-30.18/2021|Ochotona_sp.|China|Aug-2021|S|?|tile_2,0.000000,0.000000,0.000000,0.000000,0.731864,1.061005,0.197031,0.000000,0.467375,0.000000,...,0.143203,0.279667,0.000000,0.243810,0.656448,0.000000,0.000000,0.141788,0.155336,0.340542
WFG38034.1|precursor|Aba-Mianyang_virus|SC/C3-30.18/2021|Ochotona_sp.|China|Aug-2021|S|?|tile_3,0.855739,0.000000,1.338631,0.000000,0.243955,0.000000,2.655308,0.762587,0.808715,1.015744,...,0.455702,0.279667,0.000000,0.255218,0.218816,0.835083,2.154820,1.104524,0.141788,0.000000
WFG38034.1|precursor|Aba-Mianyang_virus|SC/C3-30.18/2021|Ochotona_sp.|China|Aug-2021|S|?|tile_4,0.000000,0.235260,0.000000,0.200764,0.000000,0.215085,1.951862,0.000000,1.078287,0.145106,...,0.169296,0.480624,0.624368,0.000000,0.364021,0.000000,0.000000,0.470167,0.708939,0.000000
WFG38034.1|precursor|Aba-Mianyang_virus|SC/C3-30.18/2021|Ochotona_sp.|China|Aug-2021|S|?|tile_5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
AAC04243.1|SB|Henipavirus_hendraense|isolate=?|?|?|?|segment=?|strain=?|tile_1,0.000000,0.235260,0.000000,0.000000,0.000000,0.000000,0.197031,0.000000,0.539144,0.000000,...,0.169296,0.000000,0.000000,0.638045,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AAC04243.1|SB|Henipavirus_hendraense|isolate=?|?|?|?|segment=?|strain=?|tile_2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.218816,0.533579,0.000000,0.000000,0.000000,0.000000
AAC04244.1|X|Henipavirus_hendraense|isolate=?|?|?|?|segment=?|strain=?|tile_1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.269572,0.000000,...,0.000000,0.000000,0.162931,0.000000,0.473429,0.255218,0.499895,0.000000,0.141788,0.000000
